# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~2 h on 8×A10G)

---

Previously in Part 3, we turned the training data into tokens. Now we pretrain the foundation model: a transformer that reads the token stream and learns to predict the next token, the same way an LLM learns to predict the next word. The transactions themselves are the training signal — fraud labels play no part in this step.

We start by loading the three files Part 3 wrote, then Ray Train runs the training loop — one CPU worker at `mini`, eight GPUs at `full` — and we finish by saving the trained model to shared storage.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import logging
import ray
ray.init(ignore_reinit_error=True, logging_level=logging.ERROR,
         runtime_env={"working_dir": DEMO_ROOT,
                      # torch's native JIT wants a C compiler the workers don't have
                      "env_vars": {"TORCH_DISABLE_NATIVE_JIT": "1"}})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = 8×A10G, ~2h
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

## Load the training data

Part 3 wrote three files, and this cell loads them for training.

- `ids.npy` holds the tokens as one big integer array. Each row is one window from Part 3, a stretch of one card's history.
- `attn.npy` is the attention mask: it tells the model which positions to pay attention to (1) and which to ignore (0). A card's last window is usually not full, and the empty positions are the ones marked 0.
- `vocab.json` is the token list. The trainer reads it to size the model.

We wrap the two arrays into one Ray dataset. Ray Train splits it across the training workers.

In [2]:
import numpy as np

vocab_path = os.path.join(paths["nvcorpus"], "vocab.json")
ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"))
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"))
n_seqs = int(ids.shape[0])

# Wrap the two arrays as one Ray dataset with the column names the training
# function expects. from_numpy makes a dataset out of an array; zip joins the
# two datasets column-wise.
train_ds = (ray.data.from_numpy(ids).rename_columns({"data": "input_ids"})
            .zip(ray.data.from_numpy(attn).rename_columns({"data": "attention_mask"})))

print(f"{n_seqs:,} windows × {ids.shape[1]} tokens each")
print(f"window 0, first 13 tokens as the model sees them: {ids[0][:13].tolist()}")
print("(the same window Part 3 decoded into AMT_3 MERCH_667 CAT_RETAIL ...)")

(autoscaler +9s) [autoscaler] [8cpu-32gb] Attempting to add 1 node to the cluster (increasing from 0 to 1).


(autoscaler +44s) [autoscaler] [8cpu-32gb|m5.2xlarge] [us-west-2a] [on-demand] Launched 1 instance.


8 windows × 256 tokens each
window 0, first 13 tokens as the model sees them: [1, 8, 679, 2022, 2080, 2142, 2166, 2176, 2179, 2191, 3110, 3198, 3251]
(the same window Part 3 decoded into AMT_3 MERCH_667 CAT_RETAIL ...)


## What we're training

The model is a transformer from the Llama family — the same architecture behind most open LLMs — at NVIDIA's exact configuration: 29M parameters at `full`, a 2-layer miniature at `mini`. The details are in `src/model.py`.

Training is next-token prediction. At every position, the model reads the tokens so far and predicts the one that comes next; the difference between its guess and the real token is the training signal. This is what makes the model *causal*: every prediction uses only the past. Every position in a window is one prediction, so a single 4096-token window is ~4,000 training examples. Part 5 will use the trained model's internal state as the embedding; this notebook only trains it.

## Train with Ray Train

We train with an ordinary PyTorch loop: read a batch, forward pass, backward pass, optimizer step. Four details are helpers in `src/pretrain.py`, and each is about fifteen lines including its comments: the next-token loss, the epoch metrics, the checkpoint packaging, and the cosine learning-rate schedule. Everything else is on the page.

The loop is adapted for Ray in three places, marked in the code as Ray Notes #1–#3: (#1) `prepare_model` wraps the PyTorch model for distributed training, (#2) the batches come streaming from `get_dataset_shard` instead of a DataLoader, and (#3) the loop is handed to `TorchTrainer` — explained at the run cell below.

### Defining the training function

In [3]:
import torch
from src.model import build_model
from src.pretrain import (make_lr_scheduler, next_token_loss, epoch_summary,
                          build_epoch_checkpoint)

def train_func(config: dict):
    """Runs on every training worker. Plain PyTorch plus the numbered Ray calls."""
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    torch.manual_seed(config.get("seed", 0))    # fixed seed: every worker builds identical starting weights

    # Build the model using the vocabulary created during tokenizing. 
    model = build_model(config["vocab_path"], arch=config["arch"], max_len=config["max_len"])

    # There are two two ways to train across GPUs, depending on whether the model fits
    # on one GPU. FSDP lets us split the model across gpu workers. 
    # Ray distributes the base pytorch model either way.
    if config.get("use_fsdp", False) and torch.cuda.is_available():
        from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
        model = FSDP(model.to(ray.train.torch.get_device()))
    else:
        model = ray.train.torch.prepare_model(model)    #  See Ray Note #1 above

    # Same as NVIDIA's blueprint, basic stuff for transformers.
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"],
                                  betas=tuple(config.get("betas", (0.9, 0.999))),
                                  weight_decay=float(config.get("weight_decay", 0.0)))
    scheduler = make_lr_scheduler(optimizer, config)

    # Ray splits the dataset evenly across the workers.
    # get_dataset_shard returns this worker's share — See Ray Note #2 above
    train_shard = ray.train.get_dataset_shard("train")

    # The loop itself is plain PyTorch. 
    # Ray works behind the scenes: 
    # - iter_torch_batches streams this worker's shard in as torch tensors
    # - loss.backward() averages gradients across all workers (via
    #   prepare_model wrapper), so every model copy stays identical.
    for epoch in range(config["epochs"]):
        model.train()
        running, n_batches = 0.0, 0
        for batch in train_shard.iter_torch_batches(batch_size=config["batch_size"], dtypes=torch.long):
            loss = next_token_loss(model, batch)    # the next-token prediction loss for this batch
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            running += float(loss.item())
            n_batches += 1

        metrics = epoch_summary(epoch, running / max(n_batches, 1), optimizer, config)
        ray.train.report(metrics, checkpoint=build_epoch_checkpoint(model, epoch, config))

### Run the training

In the last coding section we built the PyTorch training function, integrated with Ray for distributed training. Now we configure the training run, and run it.

Ray runs the training, distributes the data and the compute. Ray autoscales the worker group — up to eight A10G GPUs for the `full` run. And Ray makes the run durable: if training is interrupted — a node dies, a job gets killed — it resumes from the last checkpoint instead of starting over.

Ray Train reads data streamed in by Ray Data. The dataset we built at the top of the notebook goes straight into the trainer, and batches stream to each worker as training consumes them. A worker never needs the whole training set in memory, so the same code feeds a 1 GB training set or a 100 GB one.

`ScalingConfig` configures one CPU worker at `mini`, eight GPU workers at `full`. Eight workers will finish an epoch roughly eight times faster. The run takes a few minutes at `mini`, and about two hours on the eight GPUs at `full`.

In [4]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import count_schedule_steps, save_checkpoint

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]

# Calculate training steps for the learning rate schedule.
total_steps, warmup_steps = count_schedule_steps(n_seqs, pt)

# Ray Train's TorchTrainer runs train_func on every worker. 
trainer = TorchTrainer(
    train_func,
    
    # Everything train_func reads, straight from configs/<scale>.yaml.
    train_loop_config={
        "vocab_path": vocab_path, "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "seed": 0,
        "weight_decay": pt.get("weight_decay", 0.0), "betas": tuple(pt.get("betas", (0.9, 0.999))),
        "lr_schedule": pt.get("lr_schedule"), "total_steps": total_steps,
        "warmup_steps": warmup_steps, "min_lr_ratio": pt.get("min_lr_ratio", 0.0),
    },
    
    # ScalingConfig sets the worker count and GPU use, read from configs/<scale>.yaml.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    
    # Pass the Ray dataset from the load cell to Ray Train, which streams
    # a shard of it to every worker.
    datasets={"train": train_ds},
    run_config=RunConfig(
        name=f"transaction_fm_pretrain_{SCALE}",         # separate run state per scale
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage, reachable by every worker
    ),
)
result = trainer.fit()                                 # runs the whole training; returns when done
# fit() returns Ray's Result: the final metrics plus a checkpoint in Ray's run
# storage. save_checkpoint copies the weights out to the path Part 5 reads.
save_checkpoint(result, paths["checkpoint"])
print(f"done — final lm_loss {result.metrics['lm_loss']:.3f}, "
      f"perplexity {result.metrics['perplexity']:.1f} -> {paths['checkpoint']}")

(RayTrainWorker pid=3041, ip=10.0.7.12) [pretrain] epoch 1/2  lm_loss=8.742  ppl=6262.4  lr=3.00e-04
(RayTrainWorker pid=3041, ip=10.0.7.12) [pretrain] epoch 2/2  lm_loss=8.668  ppl=5816.5  lr=3.00e-04
(RayTrainWorker pid=3041, ip=10.0.7.12) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini/checkpoint_2026-07-23_17-31-56.795647)
done — final lm_loss 8.668, perplexity 5816.5 -> /mnt/cluster_storage/transaction-fm/model/mini/


## Read the training results

**Loss** measures how wrong the model's next-token guesses are — zero would mean every guess exactly right. **Perplexity** re-expresses the loss as a count — how many tokens the model is effectively choosing between at each position. Both print every epoch. Random guessing over our 6,251-token vocabulary scores a perplexity of 6,251, and repeated fields (customer id, card index, state) make the first stretch of learning easy. The trained `full` model ends near **1.7** — fewer than two effective choices per token.

A `mini` run is judged differently. Two optimizer steps cannot teach the model anything, so ignore the perplexity and check that the machinery worked — the loss fell between the two epochs, and the checkpoint reached shared storage. Part 6 scores the `full` model on actual fraud detection.

In [5]:
m = result.metrics
print(f"final causal-LM loss {m['lm_loss']:.3f}   ·   perplexity {m['perplexity']:.1f}")

final causal-LM loss 8.668   ·   perplexity 5816.5


## Save the model for the later parts

We save the trained weights to shared storage in Hugging Face format. Every later part — the embedding extraction, the fine-tune, the serving endpoint — loads the model from this one directory.

In [6]:
from src.pretrain import export_hf_model

# NVIDIA's embedding code loads models with AutoModelForCausalLM.from_pretrained,
# so we save in Hugging Face format. (export_hf_model is 14 lines in src.)
export_hf_model(paths["checkpoint"], paths["hf"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

exported HF decoder -> /mnt/cluster_storage/transaction-fm/model_hf/mini/  (state_dict: missing 0, unexpected 0)


## Scaling factors

Training cost is arithmetic: tokens × epochs × model size sets the work, and workers divide the wall-clock. Each worker trains on its own shard and Ray averages the gradients after every step. At `full` the work is 64,335 windows × 4096 tokens × 8 epochs through the 29M-parameter model — ~16,000 optimizer steps, about two hours on 8×A10G.

Memory is the constraint that mattered in practice. The loss needs a prediction for every position — a (batch × 4096 × 6,251) tensor — and at batch 8 per worker that tensor alone overflowed the A10G's 15.6 GiB. `full` runs batch 4 with gradient checkpointing (recompute activations during backward instead of storing them all), which keeps the peak near 9 GiB. The fix is in the scale config, not the training loop. If the model itself ever outgrows one GPU, `use_fsdp` splits it across workers — at 29M parameters it fits easily.

The GPU workers exist for the two hours of this stage and scale back to zero afterward.

## Takeaways

We trained the foundation model: next-token prediction over the token windows, checkpointed to shared storage, and saved in Hugging Face format for the later parts. The training function is plain PyTorch; Ray Train supplied the workers, the data sharding, the gradient averaging, and the checkpointing, and `ScalingConfig` is the only difference between one CPU worker at `mini` and eight GPUs at `full`.

This is also the one stage NVIDIA's repo skips: their notebook trains a 30-step demonstration and downloads the real weights. Our ~16,000-step run is the real training — and the Part 1 results table shows what it is worth.

---

## Next

**Part 5 — Extract embeddings**: run every transaction through the trained model to get one embedding per transaction — CPU workers feed batches to GPU workers.